# Notebook note

This notebook is the theory scratchpad behind the mean-field figures. If you only want to remake the plots, use `python scripts/make_figures.py --all`; rerun this notebook when you want to change the recursions, fits, or scaling ranges.


# Mean-field dropout criticality

This notebook evaluates the wide-network correlation recursion directly for tanh and ReLU. It is where I check the critical powers, the dropout-field exponents, and the collapse plots used in the paper.

The main quantities are the correlation length $\xi$, the decorrelation order parameter $m_*=1-c_*$, the critical relaxation $m_\ell$, and the dropout-field scaling laws.


In [ ]:
from pathlib import Path
import sys

for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "src" / "dropout_mft").exists():
        sys.path.insert(0, str(_root / "src"))
        break

import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from scipy.optimize import brentq
import warnings
warnings.filterwarnings("ignore")

# Settings
GH_N = 60  # Gauss-Hermite quadrature order
KAPPA = 2 * np.sqrt(2) / (3 * np.pi)  # ReLU kink coefficient

# Plot style
plt.rcParams.update({
    "font.family": "serif", "font.size": 11, "axes.labelsize": 13,
    "axes.titlesize": 13, "legend.fontsize": 9, "figure.dpi": 150,
    "text.usetex": False, "mathtext.fontset": "cm", "axes.linewidth": 0.8,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linestyle": "--",
})

from dropout_mft.style import COLORS, FIELD_PALETTE

PALETTE = FIELD_PALETTE

print(f"Using GH_N = {GH_N} (Gauss-Hermite quadrature)")
print(f"ReLU kink coefficient κ = {KAPPA:.8f}")

## Core Mean-Field Functions

In [ ]:
# Gauss-Hermite quadrature nodes and weights
_x, _w = hermgauss(GH_N)
_z = np.sqrt(2.0) * _x
_w1 = _w / np.sqrt(np.pi)
_Z1, _Z2 = _z[:, None], _z[None, :]
_W2 = (_w[:, None] * _w[None, :]) / np.pi

def gh_expect_1d(fvals):
    """E[f(z)] for z ~ N(0,1)."""
    return float(np.sum(_w1 * fvals))

def gh_expect_2d(fvals):
    """E[f(z1,z2)] for iid z1, z2 ~ N(0,1)."""
    return float(np.sum(_W2 * fvals))

# Tanh activation and derivatives
def tanh_phi(x): return np.tanh(x)
def tanh_phi_deriv(x): return 1.0 / np.cosh(x)**2
def tanh_phi_deriv2(x): return -2.0 * np.tanh(x) * tanh_phi_deriv(x)

# Root finding
def find_root(func, m0):
    """Find root with adaptive bracketing."""
    m_lo, m_hi = max(1e-12, m0/100), min(1.9, m0*100)
    f_lo, f_hi = func(m_lo), func(m_hi)

    for _ in range(80):
        if f_lo * f_hi < 0: break
        m_lo, m_hi = max(1e-12, m_lo/2), min(1.9, m_hi*2)
        f_lo, f_hi = func(m_lo), func(m_hi)
        if m_lo <= 1e-12 and m_hi >= 1.9: break

    if f_lo * f_hi > 0:
        ms = np.logspace(-12, 0, 800)
        fs = np.array([func(m) for m in ms])
        idx = np.where(fs[:-1] * fs[1:] < 0)[0]
        if len(idx) == 0: return m0
        m_lo, m_hi = ms[idx[0]], ms[idx[0] + 1]

    return brentq(func, m_lo, m_hi, xtol=1e-14, rtol=1e-12, maxiter=500)

# Fitting utilities
def loglog_fit(x, y, fit_slice=slice(None)):
    """Fit log₁₀(y) = a·log₁₀(x) + b."""
    x, y = np.asarray(x)[fit_slice], np.asarray(y)[fit_slice]
    mask = (x > 0) & (y > 0) & np.isfinite(x) & np.isfinite(y)
    coeffs, cov = np.polyfit(np.log10(x[mask]), np.log10(y[mask]), 1, cov=True)
    return float(coeffs[0]), float(coeffs[1]), float(np.sqrt(cov[0,0])), float(np.sqrt(cov[1,1]))

def power_law(x, a, b):
    return (10.0**b) * np.asarray(x)**a

def style_log_axes(ax, legend_loc=None):
    ax.grid(True, which='major', alpha=0.30, ls='-', lw=0.8, color=COLORS['grid'])
    ax.grid(True, which='minor', alpha=0.15, ls=':', lw=0.5, color=COLORS['grid'])
    ax.set_axisbelow(True)
    if legend_loc:
        ax.legend(loc=legend_loc, frameon=True, fancybox=True, shadow=True, framealpha=0.95)

### Tanh Mean-Field Functions

In [ ]:
def solve_qstar_tanh(sigma_w2, sigma_b2, rho, tol=1e-13, max_iter=5000):
    """Solve variance fixed point: q = (σ²_w/ρ) E[tanh(√q z)²] + σ²_b"""
    q = sigma_b2 + 1.0
    for _ in range(max_iter):
        Ef2 = gh_expect_1d(tanh_phi(np.sqrt(max(q, 0)) * _z)**2)
        q_new = (sigma_w2 / rho) * Ef2 + sigma_b2
        if abs(q_new - q) < tol * max(1.0, q_new): return float(q_new)
        q = 0.5 * (q + q_new)
    return float(q)

def tanh_chi_g_h(sigma_w2, sigma_b2, rho):
    """Return (q*, χ, g, h) where χ=F'(1), g=F''(1), h=1-F(1)."""
    q = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    sq = np.sqrt(q)
    Ef2 = gh_expect_1d(tanh_phi(sq * _z)**2)
    Efp2 = gh_expect_1d(tanh_phi_deriv(sq * _z)**2)
    Efpp2 = gh_expect_1d(tanh_phi_deriv2(sq * _z)**2)
    chi, g = sigma_w2 * Efp2, sigma_w2 * q * Efpp2
    h = 1.0 - (sigma_w2 * Ef2 + sigma_b2) / q
    return q, chi, g, h

def F_tanh(c, sigma_w2, sigma_b2, rho, qstar=None):
    """Correlation map: F(c) = (σ²_w E[φ(u₁)φ(u₂)] + σ²_b) / q*"""
    if qstar is None: qstar = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    c = float(np.clip(c, -1 + 1e-15, 1 - 1e-15))
    s, sq = np.sqrt(max(0, 1 - c*c)), np.sqrt(qstar)
    u1, u2 = sq * _Z1, sq * (c * _Z1 + s * _Z2)
    return float((sigma_w2 * gh_expect_2d(tanh_phi(u1) * tanh_phi(u2)) + sigma_b2) / qstar)

def Fp_tanh(c, sigma_w2, sigma_b2, rho, qstar=None):
    """Derivative: F'(c) = σ²_w E[φ'(u₁)φ'(u₂)]"""
    if qstar is None: qstar = solve_qstar_tanh(sigma_w2, sigma_b2, rho)
    c = float(np.clip(c, -1 + 1e-15, 1 - 1e-15))
    s, sq = np.sqrt(max(0, 1 - c*c)), np.sqrt(qstar)
    u1, u2 = sq * _Z1, sq * (c * _Z1 + s * _Z2)
    return float(sigma_w2 * gh_expect_2d(tanh_phi_deriv(u1) * tanh_phi_deriv(u2)))

def cstar_tanh(sigma_w2, sigma_b2, rho):
    """Solve F(c) = c for the stable fixed point."""
    q, chi, g, h = tanh_chi_g_h(sigma_w2, sigma_b2, rho)
    if abs(h) < 1e-15 and chi - 1 <= 0: return 1.0
    def g_m(m): return F_tanh(1-m, sigma_w2, sigma_b2, rho, qstar=q) - (1-m)
    disc = (chi-1)**2 + 2*g*h
    m0 = float(np.clip((chi-1 + np.sqrt(max(disc, 0))) / max(g, 1e-16), 1e-8, 1.0))
    return 1.0 - float(find_root(g_m, m0))

def tune_sigw2_for_chi_tanh(target_chi, sigma_b2, rho):
    def f(sw2): return tanh_chi_g_h(sw2, sigma_b2, rho)[1] - target_chi
    return brentq(f, 0.05, 20.0, xtol=1e-14, rtol=1e-12)

### ReLU Mean-Field Functions

In [ ]:
def F_relu_base(c):
    """Arc-cosine kernel (order 1) for ReLU."""
    c = float(np.clip(c, -1, 1))
    return float((np.sqrt(max(0, 1 - c*c)) + (np.pi - np.arccos(c)) * c) / np.pi)

def F_relu(c, chi, rho):
    """ReLU correlation map with tunable χ and dropout ρ."""
    return float(chi * F_relu_base(c) + 1.0 - chi / rho)

def Fp_relu(c, chi, rho):
    """Derivative: F'(c) = χ (1 - arccos(c)/π)"""
    c = float(np.clip(c, -1 + 1e-15, 1 - 1e-15))
    return float(chi * (1.0 - np.arccos(c) / np.pi))

def cstar_relu(chi, rho):
    """Solve F(c) = c for the stable fixed point."""
    h, t = 1.0 - F_relu(1.0, chi, rho), chi - 1.0
    if abs(h) < 1e-15 and t <= 0: return 1.0
    def g_m(m): return F_relu(1-m, chi, rho) - (1-m)
    m0_h = (h / (max(chi, 1e-12) * KAPPA))**(2/3) if h > 0 else 0.0
    m0_t = (t / (max(chi, 1e-12) * KAPPA))**2 if t > 0 else 0.0
    m0 = float(np.clip(max(m0_h, m0_t, 1e-8), 1e-8, 1.0))
    return 1.0 - float(find_root(g_m, m0))

## Part I: Critical Exponents (No Dropout)

We compute:
1. $\xi \sim \epsilon^{-\nu}$ as $\chi \to 1^-$
2. $m_* \sim (\chi-1)^\beta$ as $\chi \to 1^+$
3. $m_\ell \sim \ell^{-p}$ at $\chi = 1$

In [ ]:
sigma_b2 = 0.1  # Fixed bias variance

# Find critical σ²_w where χ = 1
sigw2_c = brentq(lambda sw2: tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] - 1.0, 0.1, 10.0, xtol=1e-14)
q_c, chi_c, g_c, h_c = tanh_chi_g_h(sigw2_c, sigma_b2, 1.0)
print(f"Critical σ²_w = {sigw2_c:.8f}, χ = {chi_c:.8f}, q* = {q_c:.6f}")

# (a) Correlation length ν
t_below = -np.logspace(-2, -1, 20)
xi = -1.0 / np.log(1.0 + t_below)
a_xi, b_xi, se_nu, _ = loglog_fit(np.abs(t_below), xi)
nu = -a_xi

# (b) Order parameter β
sigw2_grid = sigw2_c * (1 + np.logspace(-2, -1, 20))
chi_t = np.array([tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] for sw2 in sigw2_grid if tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] > 1])
m_t = np.array([1 - cstar_tanh(sw2, sigma_b2, 1.0) for sw2 in sigw2_grid if tanh_chi_g_h(sw2, sigma_b2, 1.0)[1] > 1])
t_t = chi_t - 1

chi_r = 1 + np.logspace(-2, -1, 20)
m_r = np.array([1 - cstar_relu(chi, 1.0) for chi in chi_r])
t_r = chi_r - 1

fit_slice = slice(0, max(6, len(t_t)//2))
a_bt, b_bt, se_bt, _ = loglog_fit(t_t, m_t, fit_slice)
a_br, b_br, se_br, _ = loglog_fit(t_r, m_r, fit_slice)

# (c) Critical relaxation p
Lmax = 20000
c = 0.0
m_l_tanh = np.empty(Lmax)
for ell in range(Lmax):
    c = F_tanh(c, sigw2_c, sigma_b2, 1.0, q_c)
    m_l_tanh[ell] = 1 - c

c = 0.0
m_l_relu = np.empty(Lmax)
for ell in range(Lmax):
    c = F_relu_base(c)
    m_l_relu[ell] = 1 - c

ells = np.arange(1, Lmax + 1)
fit_win = (ells > 500) & (ells < 8000)
a_pt, b_pt, se_pt, _ = loglog_fit(ells[fit_win], m_l_tanh[fit_win])
a_pr, b_pr, se_pr, _ = loglog_fit(ells[fit_win], m_l_relu[fit_win])

print("\n" + "="*60)
print("Part I: Critical exponents (no dropout)")
print("="*60)
print(f"  ν (correlation length):  {nu:.4f} ± {se_nu:.4f}")
print(f"  β (tanh order param):    {a_bt:.4f} ± {se_bt:.4f}")
print(f"  β (ReLU order param):    {a_br:.4f} ± {se_br:.4f}")
print(f"  p (tanh relaxation):     {-a_pt:.4f} ± {se_pt:.4f}")
print(f"  p (ReLU relaxation):     {-a_pr:.4f} ± {se_pr:.4f}")

## Part II: Dropout Perturbation Exponents

At the correlation edge ($\chi = 1$), we measure:
- $m_* \sim h^{1/\delta}$
- $\xi \sim h^{-\nu_\rho}$

In [ ]:
p_list = np.concatenate([np.logspace(-3, -2, 8), np.logspace(-2, np.log10(0.2), 15)])
rho_list = np.clip(1 - p_list, 1e-6, 1 - 1e-12)

# Smooth (tanh) at χ = 1
m_s, xi_s, h_s = [], [], []
for rho in rho_list:
    sw2 = tune_sigw2_for_chi_tanh(1.0, sigma_b2, rho)
    q, chi, g, h = tanh_chi_g_h(sw2, sigma_b2, rho)
    cstar = cstar_tanh(sw2, sigma_b2, rho)
    lam = Fp_tanh(cstar, sw2, sigma_b2, rho, q)
    m_s.append(1 - cstar)
    xi_s.append(-1 / np.log(lam))
    h_s.append(h)
m_s, xi_s, h_s = np.array(m_s), np.array(xi_s), np.array(h_s)

# Kinked (ReLU) at χ = 1
m_k, xi_k, h_k = [], [], []
for rho in rho_list:
    h = 1 - F_relu(1.0, 1.0, rho)
    cstar = cstar_relu(1.0, rho)
    lam = Fp_relu(cstar, 1.0, rho)
    m_k.append(1 - cstar)
    xi_k.append(-1 / np.log(lam))
    h_k.append(h)
m_k, xi_k, h_k = np.array(m_k), np.array(xi_k), np.array(h_k)

# Fit in smallest h decade
h_fit_max = 10 * min(h_s.min(), h_k.min())
fit_s, fit_k = h_s <= h_fit_max, h_k <= h_fit_max

a_ms, b_ms, se_ms, _ = loglog_fit(h_s[fit_s], m_s[fit_s])
a_mk, b_mk, se_mk, _ = loglog_fit(h_k[fit_k], m_k[fit_k])
a_xs, b_xs, se_xs, _ = loglog_fit(h_s[fit_s], xi_s[fit_s])
a_xk, b_xk, se_xk, _ = loglog_fit(h_k[fit_k], xi_k[fit_k])

print("\n" + "="*60)
print("Part II: Dropout perturbation exponents (small h fits)")
print("="*60)
print(f"  Smooth (tanh):")
print(f"    1/δ = {a_ms:.4f} ± {se_ms:.4f}  (δ = {1/a_ms:.3f} ± {se_ms/a_ms**2:.3f})")
print(f"    ν_ρ = {-a_xs:.4f} ± {se_xs:.4f}")
print(f"  Kinked (ReLU):")
print(f"    1/δ = {a_mk:.4f} ± {se_mk:.4f}  (δ = {1/a_mk:.3f} ± {se_mk/a_mk**2:.3f})")
print(f"    ν_ρ = {-a_xk:.4f} ± {se_xk:.4f}")

## Combined Figure: Critical Exponents and Dropout Scaling

In [ ]:
fig = plt.figure(figsize=(22, 12))
fig.patch.set_facecolor('white')
gs = fig.add_gridspec(2, 6, height_ratios=[1, 1], hspace=0.35, wspace=0.35)

ax1, ax2, ax3 = fig.add_subplot(gs[0, 0:2]), fig.add_subplot(gs[0, 2:4]), fig.add_subplot(gs[0, 4:6])
ax4, ax5 = fig.add_subplot(gs[1, 0:3]), fig.add_subplot(gs[1, 3:6])

# (a) Correlation length
t_abs = np.abs(t_below)
ax1.loglog(t_abs, xi, 'o', color=COLORS['smooth'], ms=6, mew=0.5, mec='white')
ax1.loglog(t_abs, power_law(t_abs, a_xi, b_xi), '--', color=COLORS['fit'], lw=2.5,
           label=fr"$\nu = {nu:.3f} \pm {se_nu:.3f}$")
ax1.set_xlabel(r"$|t| = |\chi - 1|$"); ax1.set_ylabel(r"$\xi = -1/\ln(\chi)$")
style_log_axes(ax1, 'lower left')

# (b) Order parameter
ax2.loglog(t_t, m_t, 'o', color=COLORS['smooth'], ms=6, mew=0.5, mec='white', label='Smooth tanh')
ax2.loglog(t_r, m_r, 's', color=COLORS['kink'], ms=6, mew=0.5, mec='white', label='Kinked ReLU')
tline = np.logspace(np.log10(t_t.min()), np.log10(t_t.max()), 200)
ax2.loglog(tline, power_law(tline, a_bt, b_bt), '--', color=COLORS['fit'], lw=2.5,
           label=fr"$\beta_{{\mathrm{{tanh}}}} = {a_bt:.3f} \pm {se_bt:.3f}$")
ax2.loglog(tline, power_law(tline, a_br, b_br), '--', color=COLORS['theory'], lw=2.5,
           label=fr"$\beta_{{\mathrm{{ReLU}}}} = {a_br:.3f} \pm {se_br:.3f}$")
ax2.set_xlabel(r"$t = \chi - 1$"); ax2.set_ylabel(r"$m_* = 1 - c_*$")
style_log_axes(ax2, 'upper left')

# (c) Critical relaxation
ax3.loglog(ells, m_l_tanh, color=COLORS['smooth'], lw=1.5, alpha=0.7, label='Smooth tanh')
ax3.loglog(ells, m_l_relu, color=COLORS['kink'], lw=1.5, alpha=0.7, label='Kinked ReLU')
ax3.loglog(ells, power_law(ells, a_pt, b_pt), '--', color=COLORS['fit'], lw=2.5,
           label=fr"$p_{{\mathrm{{tanh}}}} = {-a_pt:.3f} \pm {se_pt:.3f}$")
ax3.loglog(ells, power_law(ells, a_pr, b_pr), '--', color=COLORS['theory'], lw=2.5,
           label=fr"$p_{{\mathrm{{ReLU}}}} = {-a_pr:.3f} \pm {se_pr:.3f}$")
ax3.set_xlabel(r"Layer $\ell$"); ax3.set_ylabel(r"$m_\ell = 1 - c_\ell$")
style_log_axes(ax3, 'upper right')

# (d) Order parameter vs dropout field
hline = np.logspace(np.log10(min(h_k.min(), h_s.min())), np.log10(max(h_k.max(), h_s.max())), 200)
ax4.loglog(h_s, m_s, 'o', color=COLORS['smooth'], ms=6, mew=0.5, mec='white', label='Smooth tanh')
ax4.loglog(h_k, m_k, 's', color=COLORS['kink'], ms=6, mew=0.5, mec='white', label='Kinked ReLU')
ax4.loglog(hline, power_law(hline, a_ms, b_ms), '--', color=COLORS['fit'], lw=2.5,
           label=fr"$1/\delta_{{\mathrm{{tanh}}}} = {a_ms:.3f} \pm {se_ms:.3f}$")
ax4.loglog(hline, power_law(hline, a_mk, b_mk), '--', color=COLORS['theory'], lw=2.5,
           label=fr"$1/\delta_{{\mathrm{{ReLU}}}} = {a_mk:.3f} \pm {se_mk:.3f}$")
ax4.set_xlabel(r"Dropout field $h = 1-\bar{F}_\rho(1)$"); ax4.set_ylabel(r"$m_* = 1-c_*$")
style_log_axes(ax4, 'upper left')

# (e) Correlation length vs dropout field
ax5.loglog(h_s, xi_s, 'o', color=COLORS['smooth'], ms=6, mew=0.5, mec='white', label='Smooth tanh')
ax5.loglog(h_k, xi_k, 's', color=COLORS['kink'], ms=6, mew=0.5, mec='white', label='Kinked ReLU')
ax5.loglog(hline, power_law(hline, a_xs, b_xs), '--', color=COLORS['fit'], lw=2.5,
           label=fr"$\nu_{{\rho,\mathrm{{tanh}}}} = {-a_xs:.3f} \pm {se_xs:.3f}$")
ax5.loglog(hline, power_law(hline, a_xk, b_xk), '--', color=COLORS['theory'], lw=2.5,
           label=fr"$\nu_{{\rho,\mathrm{{ReLU}}}} = {-a_xk:.3f} \pm {se_xk:.3f}$")
ax5.set_xlabel(r"Dropout field $h$"); ax5.set_ylabel(r"$\xi = -1/\ln\lambda$")
style_log_axes(ax5, 'upper right')

plt.savefig('combined_exponents_and_dropout_scaling.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## Part III: Universal Scaling Collapse

**Smooth (tanh):** $\tilde{m} = m\sqrt{g/(2h)}, \quad \tilde{\tau} = (1-\chi)/\sqrt{2gh}$

**Kinked (ReLU):** $\tilde{m} = m/(h/\kappa)^{2/3}, \quad u = (1-\chi)/(\kappa^{2/3}h^{1/3})$

In [ ]:
h_list = np.logspace(-4, -1, 10)
t_vals = np.linspace(-0.3, 0.3, 50)

# Smooth (tanh) data
smooth_data = []
for h_proxy in h_list:
    rho = 1 / (1 + h_proxy)
    for t in t_vals:
        sw2 = tune_sigw2_for_chi_tanh(1 + t, sigma_b2, rho)
        q, chi, g, h = tanh_chi_g_h(sw2, sigma_b2, rho)
        m = 1 - cstar_tanh(sw2, sigma_b2, rho)
        smooth_data.append((h_proxy, t, h, g, m))
smooth = np.array(smooth_data)
h_proxy_s, t_s, h_s2, g_s, m_s2 = smooth.T
tau_tilde_s = -t_s / np.sqrt(2 * g_s * h_s2)
m_tilde_s = m_s2 * np.sqrt(g_s / (2 * h_s2))

# Kinked (ReLU) data
kink_data = []
for h_target in h_list:
    rho = 1 / (1 + h_target)
    for t in t_vals:
        m = 1 - cstar_relu(1 + t, rho)
        h = 1 - F_relu(1.0, 1 + t, rho)
        kink_data.append((h_target, t, h, m))
kink = np.array(kink_data)
h_target_k, t_k, h_k2, m_k2 = kink.T
u_k = -t_k / (KAPPA**(2/3) * h_k2**(1/3))
m_tilde_k = m_k2 / ((h_k2 / KAPPA)**(2/3))

# Theory curves
tau_line = np.linspace(-6, 6, 600)
m_smooth_theory = np.sqrt(1 + tau_line**2) - tau_line

def kink_universal(u):
    out = np.empty_like(np.atleast_1d(u), dtype=float)
    for i, ui in enumerate(np.atleast_1d(u)):
        roots = np.roots([1, ui, 0, -1])
        real = roots[np.isclose(roots.imag, 0, atol=1e-10)].real
        real = real[real > 0]
        out[i] = real.max()**2 if len(real) else np.nan
    return out

u_line = np.linspace(-6, 6, 600)
m_kink_theory = kink_universal(u_line)

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(12, 8))
fig.patch.set_facecolor('white')

def plot_scatter(ax, h_vals, x, y, marker, t_arr, h_arr):
    for idx, hp in enumerate(h_list):
        mask = np.isclose(h_vals, hp)
        mask0 = mask & np.isclose(t_arr, 0.0, atol=1e-12)
        h0 = float(np.median(h_arr[mask0])) if mask0.any() else float(np.median(h_arr[mask]))
        ax.plot(x[mask], y[mask], marker, ms=5, color=PALETTE[idx],
               mew=0.5, mec='white', label=fr"$h\approx {h0:.0e}$", alpha=0.9)

# (a) Smooth raw
plot_scatter(axs[0,0], h_proxy_s, t_s, m_s2, 'o', t_s, h_s2)
axs[0,0].set_xlabel(r"$t=\chi-1$"); axs[0,0].set_ylabel(r"$m=1-c_*$")
axs[0,0].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper left')

# (b) Smooth collapse
plot_scatter(axs[0,1], h_proxy_s, tau_tilde_s, m_tilde_s, 'o', t_s, h_s2)
axs[0,1].plot(tau_line, m_smooth_theory, '-', color='#2D2D2D', lw=2.5, label="Theory", zorder=10)
axs[0,1].set_xlabel(r"$\tilde{\tau}=(1-\chi)/\sqrt{2gh}$"); axs[0,1].set_ylabel(r"$\tilde{m}=m\sqrt{g/(2h)}$")
axs[0,1].set_xlim(-1, 2); axs[0,1].set_ylim(0, 2)
axs[0,1].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper right')

# (c) Kinked raw
plot_scatter(axs[1,0], h_target_k, t_k, m_k2, 's', t_k, h_k2)
axs[1,0].set_xlabel(r"$t=\chi-1$"); axs[1,0].set_ylabel(r"$m=1-c_*$")
axs[1,0].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper left')

# (d) Kinked collapse
plot_scatter(axs[1,1], h_target_k, u_k, m_tilde_k, 's', t_k, h_k2)
axs[1,1].plot(u_line, m_kink_theory, '-', color='#2D2D2D', lw=2.5, label="Theory", zorder=10)
axs[1,1].set_xlabel(r"$u=(1-\chi)/(\kappa^{2/3}h^{1/3})$"); axs[1,1].set_ylabel(r"$m/(h/\kappa)^{2/3}$")
axs[1,1].set_xlim(-1.25, 2); axs[1,1].set_ylim(0, 2)
axs[1,1].legend(frameon=True, ncol=2, fancybox=True, shadow=True, framealpha=0.95, loc='upper right')

for ax in axs.flat:
    ax.grid(True, which='major', ls='-', alpha=0.3, lw=0.8, color=COLORS['grid'])
    ax.grid(True, which='minor', ls=':', alpha=0.15, lw=0.5, color=COLORS['grid'])
    ax.set_axisbelow(True)

plt.tight_layout(pad=1.5)
plt.savefig('scaling_collapse_improved.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()